<a href="https://colab.research.google.com/github/aqmalio/Data-Science-2026/blob/main/Pertemuan12_Aqmal_250401020204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
* Nama: Aqmal
* Nim: 250401020149
* Kelas: IF405
---

### Generate & Eksplorasi Dataset Transaksi
Buat dataset transaksi sintetis dengan pola pembelian tersembunyi.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings
from datetime import datetime, timezone
import logging


# Hide DeprecationWarning
logging.captureWarnings(True)
logging.getLogger("py.warnings").setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=DeprecationWarning)

np.random.seed(42)

produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


### One-Hot Encoding Transaksi
Ubah daftar transaksi menjadi tabel one-hot encoding menggunakan `TransactionEncoder`.

In [2]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
display(df.head())

,Gula,Keju,Kopi,Mentega,Roti,Selai,Sereal,Susu,Teh,Telur
0,False,True,True,True,True,True,False,False,False,False
1,False,False,True,True,True,True,False,False,True,False
2,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,True,True
4,True,True,False,True,False,False,False,True,False,False


### Cari Frequent Itemset dengan Apriori
Jalankan Apriori dengan beberapa nilai `min_support`.

In [3]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
  print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


### Bentuk & Saring Aturan Asosiasi
Bentuk aturan asosiasi dan saring berdasarkan metrik `confidence` dan `lift`.

In [4]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

,antecedents,consequents,support,confidence,lift
9,"(Teh, Keju)",(Telur),0.12,0.857143,2.380952
13,"(Mentega, Selai)",(Kopi),0.10,0.625000,1.953125
11,"(Roti, Gula)",(Selai),0.10,1.000000,1.923077
7,(Sereal),(Mentega),0.14,0.777778,1.851852
8,"(Teh, Telur)",(Keju),0.12,0.600000,1.764706
15,"(Selai, Kopi)",(Mentega),0.10,0.714286,1.700680
10,"(Telur, Keju)",(Teh),0.12,0.750000,1.630435
12,"(Gula, Selai)",(Roti),0.10,0.500000,1.562500
14,"(Mentega, Kopi)",(Selai),0.10,0.714286,1.373626
1,(Roti),(Selai),0.22,0.687500,1.322115


#### Interpretasi

*   **Aturan Paling Kuat (Lift Tertinggi):** Berdasarkan tabel di atas, aturan `{Roti, Gula} -> {Selai}` memiliki nilai **Lift (~1.92)** dan **Confidence (1.0)** tertinggi. Ini menunjukkan hubungan asosiasi yang sangat kuat.
*   **Analisis Bisnis:** Hasil ini sangat masuk akal secara bisnis. Tingginya nilai lift pada aturan yang melibatkan `Roti` dan `Selai` membuktikan bahwa pola yang di 'suntikkan' pada tahap pembuatan dataset berhasil dideteksi oleh algoritma Apriori.

### Rekomender Sederhana dengan Content-Based Filtering
Bangun katalog produk dengan kategori dan gunakan `cosine similarity`.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy', 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


### Langkah 6: Bandingkan Kedua Pendekatan
Bandingkan rekomendasi dari Association Rules dengan Content-Based Filtering untuk produk yang sama.

In [6]:
produk_target = 'Roti'

# Rekomendasi dari Association Rules
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print('--- Rekomendasi dari Association Rules ---')
print(rules_terkait[['consequents', 'lift']].head())

print('\n--- Rekomendasi dari Content-Based ---')
print(rekomendasi_serupa(produk_target))

--- Rekomendasi dari Association Rules ---
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

--- Rekomendasi dari Content-Based ---
['Selai', 'Sereal', 'Susu']


### Ringkasan Analisis

**1. Konsistensi Rekomendasi:**
*   **Konsisten:** Keduanya menyarankan **Selai** untuk **Roti**. Association Rules menemukannya dari riwayat belanja nyata, sedangkan Content-Based menemukannya karena kesamaan kategori ('Bakery').

**2. Kapan Menggunakan Salah Satu?**
*   **Association Rules:** Gunakan saat punya data transaksi yang banyak untuk melihat hubungan antar produk yang tidak terduga.
*   **Content-Based:** Gunakan untuk **produk baru** yang belum punya sejarah transaksi (Cold Start Problem).

**3. Strategi Hybrid (Gabungan):**
*   Cara terbaik adalah menggabungkannya: Association Rules sebagai rekomendasi utama untuk *Cross-selling*, lalu lengkapi dengan Content-Based untuk menawarkan variasi produk serupa yang mungkin belum pernah dibeli pelanggan tersebut.

### Kesimpulan

**Apa yang dipelajari:**
Saya mempelajari cara mengimplementasikan dua sistem rekomendasi berbeda: *Association Rules* (Apriori) yang berbasis perilaku sosial/transaksi, dan *Content-Based Filtering* yang berbasis kemiripan sifat produk.

**Temuan Utama:**
*   Algoritma Apriori sangat efektif menangkap pola tersembunyi (seperti Roti & Selai) dengan metrik *Lift* sebagai penentu kekuatan aturan.
*   Kedua metode memberikan hasil yang konsisten untuk produk utama, namun *Content-Based* memberikan variasi lebih banyak karena melihat kesamaan kategori.

**Keterbatasan & Pertanyaan:**
*   **Keterbatasan:** Dataset yang digunakan masih sangat kecil (sintetis), sehingga nilai *support* sensitif terhadap perubahan data.
*   **Pertanyaan:** Bagaimana performa sistem ini jika menghadapi ribuan produk (*Cold Start Problem*) dan manakah yang lebih efisien secara komputasi untuk data skala besar?